# Stablecoin StressBench — Model Training

This notebook demonstrates training all benchmark models:
- Naive baselines (last value, rolling mean, AR1)
- Linear models (logistic, ridge, lasso)
- Tree models (LightGBM, XGBoost, Random Forest)
- Sequence models (Temporal CNN, Transformer)

In [ ]:
import numpy as np
import polars as pl
from pathlib import Path
from stressbench.experiments.feature_sets import FEATURE_SETS
from stressbench.models.baselines import (
    LastValueBaseline, RollingMeanBaseline, AR1Baseline,
    LogisticBaseline, RidgeBaseline, LassoBaseline,
)
from stressbench.models.tree_models import LGBMWrapper, XGBWrapper, RandomForestWrapper

DATASET_PATH = Path("../data/gold/dataset.parquet")
NET_COL      = "net_profit_bps_q10000"   # K notional net profit in bps
NET_THRESH   = 10.0                        # c = 10 bps (Eq. 5 in paper)
FEATURE_SET  = "price_plus_book"           # 12 features: 5 price + 7 book

df = pl.read_parquet(DATASET_PATH)
feature_cols = FEATURE_SETS[FEATURE_SET]

def load_split(df, split, net_col, net_thresh, feature_cols):
    """Load (X, y_exec, y_net) for a data split.
    y_exec: binary executable-arb label  (net > c, matches paper Eq. 5)
    y_net:  raw net profit in bps (for economic evaluation)
    """
    sdf = df.filter(pl.col("split") == split)
    X_raw = sdf.select(feature_cols).to_numpy().astype(np.float32)
    nan_mask = np.isnan(X_raw)
    if nan_mask.any():
        col_med = np.where(np.isnan(np.nanmedian(X_raw, axis=0)), 0.0,
                           np.nanmedian(X_raw, axis=0))
        X = np.where(nan_mask, col_med[None, :], X_raw)
    else:
        X = X_raw
    y_net = sdf[net_col].to_numpy().astype(np.float64)
    y_exec = (y_net > net_thresh).astype(np.int8)
    return X, y_exec, y_net

X_train, y_train, y_net_train = load_split(df, "train",      NET_COL, NET_THRESH, feature_cols)
X_val,   y_val,   y_net_val   = load_split(df, "validation", NET_COL, NET_THRESH, feature_cols)
X_test,  y_test,  y_net_test  = load_split(df, "test",       NET_COL, NET_THRESH, feature_cols)

# Paper Table 2: train 0.00%, val 2.30%, test 2.88% (executable arb, c=10bps)
print(f"Train: {X_train.shape},  pos rate: {y_train.mean():.4f}  (calm -- should be 0.0)")
print(f"Val:   {X_val.shape},    pos rate: {y_val.mean():.4f}  (Terra/LUNA)")
print(f"Test:  {X_test.shape},   pos rate: {y_test.mean():.4f}  (USDC/SVB)")
print(f"Features ({len(feature_cols)}): {feature_cols}")


In [ ]:
from sklearn.metrics import roc_auc_score

def _roc(y_true, y_score):
    try:
        return roc_auc_score(y_true, y_score)
    except Exception:
        return float("nan")

# Train all models on the calm-control training split
models = {
    "last_value":  LastValueBaseline(),
    "logistic":    LogisticBaseline(),
    "rf":          RandomForestWrapper(task="classification", n_estimators=100),
    "lgbm":        LGBMWrapper(task="classification"),
    "xgb":         XGBWrapper(task="classification"),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    try:
        val_proba  = model.predict_proba(X_val)[:, 1]
        test_proba = model.predict_proba(X_test)[:, 1]
        print(f"{name:15s}  val_auc={_roc(y_val, val_proba):.3f}  "
              f"test_auc={_roc(y_test, test_proba):.3f}")
    except (AttributeError, ValueError) as e:
        print(f"{name:15s}  (no predict_proba: {e})")
